# 02. 입찰메이트 RFP RAG — Phase 1 기준 확정

유영관 · 2026-11-10

이 노트북은 **업무 범위·정답 근거·평가 기준을 확인하는 노트북**입니다.
OpenAI API 호출, 임베딩, 검색, 모델 응답 생성은 수행하지 않습니다.

| 항목 | 이번 산출물 |
| --- | --- |
| 자료 | 공개 RFP 파일 100개 / 고유 내용 99개 |
| 기능 | 사실 확인 · 전체 요약 · 여러 문서 비교 · 후속 질문 · 불확실성 대응 |
| 평가 사례 | 100개: 개발용 60 / 최종 평가용 40 |
| 후속 대화 | 20개 대화, 64턴 |
| 실행 결과 | 평가셋의 구조·근거 일치 검증. RAG 성능은 미측정 |

ZIP을 풀고 같은 폴더 안의 이 노트북을 위에서부터 실행합니다. **Python 3.10 이상, 표준 라이브러리만** 사용합니다.
기존 `01_...` 노트북의 이전 10개 문서·25문항은 이번 기준에 섞지 않습니다.


## 1. 파일 위치 확인

`notebooks/` 또는 패키지 루트에서 열 수 있습니다. 폴더 이름을 바꿨다면 `ROOT`에 압축을 푼 경로를 직접 지정하세요.


In [10]:
from pathlib import Path
import json
import sys

candidates = [Path.cwd(), Path.cwd().parent,
              Path.cwd() / 'rfp_phase1_20261110',
              Path.cwd() / 'output/rfp_phase1_20261110']
ROOT = next((p.resolve() for p in candidates if (p / 'phase1_tools.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('ZIP을 푼 뒤 패키지 안에서 노트북을 열거나 ROOT를 직접 지정하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from phase1_tools import load_data, read_jsonl, validate_bundle, runtime_case

documents, sources, dev_cases = load_data(ROOT, 'dev')
doc_by_id = {d['doc_id']: d for d in documents}
print('자료 준비:', len(documents), '파일 / 개발 사례', len(dev_cases), '개')


자료 준비: 100 파일 / 개발 사례 60 개


## 2. 평가셋 검증

문서 ID·내용 해시·출처 위치·인용 원문·분할·사례 수를 검증합니다.
최종 평가 정답을 화면에 출력하지 않고 참조 무결성만 검사합니다.
**통과는 모델이 잘 답한다는 뜻이 아닙니다.**


In [11]:
validation = validate_bundle(ROOT)
assert validation['valid'], validation['errors']
print(json.dumps(validation, ensure_ascii=False, indent=2))


{
  "valid": true,
  "errors": [],
  "document_files": 100,
  "unique_documents": 99,
  "cases": 100,
  "dev_cases": 60,
  "holdout_cases": 40,
  "facts": 396,
  "evidence_links": 684,
  "feature_counts": {
    "comparison": 20,
    "followup": 20,
    "fact": 25,
    "summary": 20,
    "uncertainty": 15
  },
  "content_disjoint": true,
  "rag_performance_measured": false,
  "check_scope": "구조·참조·원문 인용 일치. 의미 지원 여부 검수와 RAG 성능 평가는 별도."
}


## 3. 원문 자료 점검

실제 파일 형식으로 보면 HWP 47개, HWPX 23개, PDF 30개입니다.
확장자와 실제 형식이 다른 1개와, 추출 중 불량 문자 1개가 치환된 1개는 Phase 2에서 다시 검증합니다.
텍스트 추출 성공만으로 표·그림의 완전한 보존을 확정하지 않습니다.


In [12]:
audit = json.loads((ROOT / 'reports/data_audit.json').read_text(encoding='utf-8'))
for key in ['file_count', 'unique_content_count', 'detected_format_counts',
            'assigned_unique_document_splits', 'year_unconfirmed_count']:
    print(key, ':', audit[key])
for item in audit['format_warnings']:
    print('파싱 확인 대상:', item['name'], item['warnings'])


file_count : 100
unique_content_count : 99
detected_format_counts : {'hwpx': 23, 'hwp': 47, 'pdf': 30}
assigned_unique_document_splits : {'dev': 60, 'holdout': 39}
year_unconfirmed_count : 3
파싱 확인 대상: 서울특별시_2026_2027년 재난정보관리시스템 유지관리 용역_제안요청서.hwpx ['extension_does_not_match_magic']
파싱 확인 대상: 한국학호남진흥원_2025_2025년 소장 국학자료 원문이미지 DB 구축 용역_제안요청서.hwp ['one_invalid_utf16_unit_replaced_in_extracted_text']


## 4. 개발용 대표 사례 10개 보기

다섯 기능에서 2개씩 선정했습니다. 질문·필수 사실·근거·합격 동작을 함께 확인합니다.
이것은 평가용 정답 데이터입니다. 이후 모델에 질문을 보낼 때 정답 부분을 넣지 않습니다.


In [13]:
smoke_cases = read_jsonl(ROOT / 'eval/questions.smoke.dev.jsonl')
print([(c['case_id'], c['feature']) for c in smoke_cases])

def show_case(case_id):
    case = next(c for c in dev_cases if c['case_id'] == case_id)
    print('사례:', case['case_id'], '/', case['feature'])
    print('질문:', case['question'])
    print('기대 상태:', case['expected_status'])
    if case.get('turns'):
        for turn in case['turns']:
            print(turn['turn_id'], turn['user'], '→', turn['expected_status'])
    for fact in case['required_facts']:
        print('정답', fact['fact_id'], fact['field'], ':', fact['expected_value'])
        for evidence in fact['evidence']:
            print('  근거:', doc_by_id[evidence['doc_id']]['name'], evidence['locator'])
            print('  인용:', evidence['quote'][:240])
    for behavior in case['required_behaviors']:
        print('합격 동작:', behavior)
    return case

sample_fact = show_case(next(c['case_id'] for c in smoke_cases if c['feature'] == 'fact'))


[('F001', 'fact'), ('F002', 'fact'), ('S001', 'summary'), ('S002', 'summary'), ('C001', 'comparison'), ('C002', 'comparison'), ('D001', 'followup'), ('D002', 'followup'), ('U001', 'uncertainty'), ('U002', 'uncertainty')]
사례: F001 / fact
질문: 2027년 글로벌 패션브랜드 육성 용역의 용역비는 얼마이며 부가세가 포함되나요?
기대 상태: answered
정답 f1 budget : 650,000,000원, 부가세 포함
  근거: 서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx Contents/section0.xml
  인용: 용 역 비 : 
650
,000,000원
(부가세 포함) 
 
 
합격 동작: 선택한 문서의 본문 근거와 위치를 제시한다.
합격 동작: 숫자 중간의 추출 줄바꿈을 제거해 650,000,000원으로 읽고 부가세 포함 여부를 명시한다.


## 5. 전체 요약의 기준

사업명·기관·목적·범위/요구사항·예산·기간·참가자격·제출서류·기한·방식·평가기준의 11개 항목입니다.
`not_established_from_reviewed_text`는 검토한 텍스트에서 확정 근거를 확보하지 못했다는 뜻입니다.
원문에 해당 조건이 없다는 확정과 구분합니다. 공고문 참조는 참조 사실 자체를 보존합니다.


In [14]:
sample_summary = show_case(next(c['case_id'] for c in smoke_cases if c['feature'] == 'summary'))
print(json.dumps(sample_summary['summary_field_expectations'], ensure_ascii=False, indent=2))


사례: S001 / summary
질문: 선택한 문서 전체를 검토하여 사업명, 발주기관, 목적, 주요 범위·필수요건, 예산과 VAT, 수행기간, 참가·수행자격, 입찰 제출서류, 제출기한, 제출방식, 평가기준의 11개 항목으로 요약해 주세요. 공고문 참조 사항과 확인되지 않는 사항도 구별해 주세요.
기대 상태: answered
정답 f1 project_name : 스마트패스 전용 출국장 운영 위탁 용역
  근거: 인천공항시설관리(주)_2026_스마트패스 전용 출국장 운영 위탁 용역_과업내용서.hwp BodyText/Section0 / paragraph 64
  인용: ㅇ 스마트패스 전용 출국장 운영 위탁 용역
정답 f2 agency : 인천공항시설관리(주)
  근거: 인천공항시설관리(주)_2026_스마트패스 전용 출국장 운영 위탁 용역_과업내용서.hwp BodyText/Section0 / paragraph 85
  인용: ㅇ‘발주자’란 인천공항시설관리(주) 및 본 과업의 담당자를 말한다.
정답 f3 purpose : 스마트패스 전용 출국장 원활한 운영으로 이용객 편의 및 안정적인 출국장 운영을 확보
  근거: 인천공항시설관리(주)_2026_스마트패스 전용 출국장 운영 위탁 용역_과업내용서.hwp BodyText/Section0 / paragraph 68
  인용: ㅇ 스마트패스 전용 출국장의 원활한 운영을 통한 이용객 편의 증진 및 안정
  근거: 인천공항시설관리(주)_2026_스마트패스 전용 출국장 운영 위탁 용역_과업내용서.hwp BodyText/Section0 / paragraph 69
  인용: 적인 출국장 운영
정답 f4 scope_and_requirements : 스마트패스 이용객·등록·오진입 안내 및 문의 응대. 매일 05:00~08:00(3시간), 운영인력 총24명 이상 확보 및 결원 즉시 대체
  근거: 인천공항시설관리(주)_2026_스마트패스 전용 출국장 운영 위탁 용역_과업내용서.hwp BodyText/Section0 / paragraph 1

## 6. 여러 문서 비교의 기준

문서 × 요청 항목의 모든 칸을 채점합니다. 연차·부가세·금액 단위가 다르면 원문 기준을 남깁니다.
원문에 실제 충돌이 있으면 한쪽 값으로 확정하지 않습니다.


In [15]:
sample_comparison = show_case(next(c['case_id'] for c in smoke_cases if c['feature'] == 'comparison'))


사례: C001 / comparison
질문: 선택한 다음 두 RFP를 비교해 줘: 「한국주택금융공사: HF 인사이트 시스템 개발」 / 「한국주택금융공사: 부동산 등기정보 시스템 구축」. 비교 항목은 예산(부가세 및 개발·HW 내역이 있으면 포함)과 사업기간(안정화·조정 조건 포함)야. 문서별 열을 두는 표로 정리하고 각 데이터 칸에 근거를 붙여 줘. 기간이나 예산의 기준이 다르면 그대로 설명하고 우열 추천은 하지 마.
기대 상태: answered
정답 f1 budget : 688백만원(부가세 포함). 개발예산 498백만원과 HW구입 190백만원으로 구성.
  근거: 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp BodyText/Section0 / paragraph 99
  인용: □ 사업예산 : 688백만원 (부가세포함)
  근거: 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp BodyText/Section0 / paragraph 100
  인용: ㅇ 개발예산 (498백만원) 및  HW구입(190백만원)
정답 f2 duration : 계약일로부터 5개월. 안정화·시스템모니터링 1개월 포함.
  근거: 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp BodyText/Section0 / paragraph 97
  인용: □ 사업기간 : 계약일로부터 5개월(안정화·시스템모니터링 1개월 포함)
정답 f3 budget : 280백만 원 이내(부가세 포함).
  근거: 한국주택금융공사_2021_부동산 등기정보 시스템 구축_제안요청서.hwp BodyText/Section0 / paragraph 122
  인용: □ 사업예산 : 280백만 원 이내(부가세 포함)
정답 f4 duration : 계약체결일로부터 365일. 외부 정보연계 등 협의 지연 시 단계적 오픈 및 과업 조정 가능.
  근거: 한국주택금융공사_2021_부동산 등기정보 시스템 구축_제안요청서.hwp BodyText/Section0 / par

## 7. 후속 질문의 기준

대화는 2~4턴으로 구성하며 모든 평가 턴의 정답과 실제 선택 범위를 확인합니다.
모호한 첫 질문에서는 선택 범위를 비우고 후보를 구분하는 질문을 합니다.
평가 실행기는 사용자 발화만 한 턴씩 전달하고, 모델이 실제 생성한 답변으로 대화를 이어갑니다.


In [16]:
sample_dialogue = show_case(next(c['case_id'] for c in smoke_cases if c['feature'] == 'followup'))
runtime_input = runtime_case(sample_dialogue)
assert 'required_facts' not in runtime_input
assert all('expected_doc_ids' not in turn for turn in runtime_input['turns'])
print('정답 제거 후 실행기 입력 필드:', list(runtime_input))


사례: D001 / followup
질문: 한국주택금융공사의 HF 인사이트 시스템 개발 예산이 얼마야? 세금 포함 여부도 알려줘.
기대 상태: answered
t1 한국주택금융공사의 HF 인사이트 시스템 개발 예산이 얼마야? 세금 포함 여부도 알려줘. → answered
t2 그건 몇 개월짜리야? 안정화도 그 안에 들어가? → answered
t3 아까 그 예산을 개발비와 장비비로 나누면? → answered
정답 f1 budget : 688백만원, 부가세 포함.
  근거: 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp BodyText/Section0 / paragraph 99
  인용: □ 사업예산 : 688백만원 (부가세포함)
정답 f2 duration : 계약일로부터 5개월이며 안정화·시스템모니터링 1개월을 포함한다.
  근거: 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp BodyText/Section0 / paragraph 97
  인용: □ 사업기간 : 계약일로부터 5개월(안정화·시스템모니터링 1개월 포함)
정답 f3 budget_components : 개발예산 498백만원과 HW구입 190백만원으로 구성한다.
  근거: 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp BodyText/Section0 / paragraph 100
  인용: ㅇ 개발예산 (498백만원) 및  HW구입(190백만원)
합격 동작: 각 턴의 expected_status, 문서 범위, 필수 사실을 독립적으로 충족한다.
합격 동작: 이전 assistant 응답 자체를 문서 근거로 사용하지 않고 해당 턴의 원문 근거를 확인해 인용한다.
합격 동작: 실행 대화 이력에는 사용자 질문과 실제 모델 응답만 넣으며 gold 사실·기대 문서 범위를 사용자 이력에 주입하지 않는다.
합격 동작: 그건·그 예산을 HF 인사이트 사업으로 유지한다.
합격 동작: 5개월과 별도 1개월로 더하지 않는다.
정답 제거 후 실행기 입력 필드

## 8. 불확실성 대응의 기준

선택이 모호하면 확인 질문, 외부 자료가 없으면 근거 부족, 일부 사실만 있으면 부분 답변으로 처리합니다.
파싱·검색·API 실패는 정보 미기재와 구별합니다. 합성 장애 사례는 가상의 도구 결과로 명시되어 있습니다.


In [17]:
sample_uncertainty = show_case(next(c['case_id'] for c in smoke_cases if c['feature'] == 'uncertainty'))


사례: U001 / uncertainty
질문: 서울시 2027년 유지관리 사업의 예산을 알려줘.
기대 상태: needs_clarification
합격 동작: 현재 문서가 선택되지 않았고 후보가 둘 이상임을 인식한다.
합격 동작: 운수사업관리시스템과 침수예측 정보시스템 후보를 구분해 제시하고 사업명을 확인한다.
합격 동작: 후보 중 하나의 예산을 임의로 답하거나 두 사업의 금액을 합치지 않는다.


## 9. 업무 관점에서 남길 의견

대표 10개 중 실제 입찰 검토에서 유용한 질문인지 확인하고 의견이 있으면 아래에 남깁니다.
이는 업무 담당자의 검토 기록입니다. 현재 정답 작성·원문 검수와 별도로 관리합니다.
비밀 문서나 API 키를 적지 않습니다.


In [18]:
business_reviews = [
    # {'case_id': 'F001', 'decision': 'keep', 'comment': '실제 업무에서 자주 묻는 질문'},
]
if business_reviews:
    review_path = ROOT / 'reports/business_review_notes.json'
    review_path.write_text(json.dumps(business_reviews, ensure_ascii=False, indent=2), encoding='utf-8')
    print('검토 의견 저장:', review_path.name)
else:
    print('업무 검토 의견은 아직 기록되지 않았습니다.')


업무 검토 의견은 아직 기록되지 않았습니다.


## 10. Phase 1 종료와 Phase 2 시작점

- 다섯 기능의 입출력·근거·실패 대응과 평가 기준을 정했습니다.
- 새 공개 자료에 근거한 평가 사례 100개를 개발용 60 / 최종 평가용 40으로 구성했습니다.
- 문서별 공개 자료 범위, 중복 처리, 실제 형식과 파싱 확인 대상을 기록했습니다.
- 정답 인용의 원문 일치와 참조 관계를 검사했습니다.

**아직 측정하지 않은 것:** 검색 정확도, 생성 답변 정확도, 처리시간, API 비용.
다음 Phase 2는 원본 PDF/HWP/HWPX의 파싱·표·출처 추적을 검증하는 단계입니다.
이 노트북의 추출 텍스트 스냅샷은 검증 기준이며, 원본 파일은 Drive의 `mid-project-rfp-public` 폴더에서 가져옵니다.
